In [1]:
import os
import glob
import numpy as np
import pandas as pd

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'

RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')

paths = glob.glob(os.path.join(RAW_DIR, '*07_scheme_performance*'))
print(paths)

['D:\\Bluestock work\\bluestock-fintech-analysis1\\data\\raw\\1788499985420-bb134abf-07_scheme_performance.csv']


In [3]:
perf = pd.read_csv(paths[0])
n_start = len(perf)

print(perf.shape)
print(perf.dtypes)
print(perf.isnull().sum())

(40, 19)
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0

In [4]:
RETURN_COLS = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct']

OTHER_NUM_COLS = ['alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
                  'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore',
                  'expense_ratio_pct', 'morningstar_rating']

NUM_COLS = RETURN_COLS + OTHER_NUM_COLS

log = []

def add_log(step, df):
    log.append({'step': step, 'rows': len(df)})

add_log('start', perf)

In [5]:
fake_missing = ['', 'NA', 'N/A', 'n/a', 'null', 'NULL', 'None', '-', '--']

for col in NUM_COLS:
    cleaned = perf[col].replace(fake_missing, np.nan)
    coerced = pd.to_numeric(cleaned, errors='coerce')
    bad = coerced.isnull() & cleaned.notnull()
    print(col, '| not numbers:', bad.sum(), '| examples:', list(cleaned[bad].unique()[:5]))

return_1yr_pct | not numbers: 0 | examples: []
return_3yr_pct | not numbers: 0 | examples: []
return_5yr_pct | not numbers: 0 | examples: []
benchmark_3yr_pct | not numbers: 0 | examples: []
alpha | not numbers: 0 | examples: []
beta | not numbers: 0 | examples: []
sharpe_ratio | not numbers: 0 | examples: []
sortino_ratio | not numbers: 0 | examples: []
std_dev_ann_pct | not numbers: 0 | examples: []
max_drawdown_pct | not numbers: 0 | examples: []
aum_crore | not numbers: 0 | examples: []
expense_ratio_pct | not numbers: 0 | examples: []
morningstar_rating | not numbers: 0 | examples: []


In [6]:
REMOVE_SYMBOLS = False   # change to True if Part A shows values like '12.5%' or '1,250'

flags = pd.DataFrame(index=perf.index)
flags['not_number'] = False
not_number_counts = {}

for col in NUM_COLS:
    cleaned = perf[col].replace(fake_missing, np.nan)
    has_value = cleaned.notnull()

    if REMOVE_SYMBOLS:
        cleaned = cleaned.astype(str).str.replace(r'[%,\s]', '', regex=True)

    coerced = pd.to_numeric(cleaned, errors='coerce')
    not_number = coerced.isnull() & has_value

    not_number_counts[col] = int(not_number.sum())
    flags['not_number'] = flags['not_number'] | not_number
    perf[col] = coerced

for col in NUM_COLS:
    print(col, not_number_counts[col])

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0
benchmark_3yr_pct 0
alpha 0
beta 0
sharpe_ratio 0
sortino_ratio 0
std_dev_ann_pct 0
max_drawdown_pct 0
aum_crore 0
expense_ratio_pct 0
morningstar_rating 0


In [7]:
hard_masks = {
    'return_1yr_pct': perf['return_1yr_pct'] < -100,
    'return_3yr_pct': perf['return_3yr_pct'] < -100,
    'return_5yr_pct': perf['return_5yr_pct'] < -100,
    'benchmark_3yr_pct': perf['benchmark_3yr_pct'] < -100,
    'max_drawdown_pct': perf['max_drawdown_pct'].abs() > 100,
    'std_dev_ann_pct': perf['std_dev_ann_pct'] < 0,
    'aum_crore': perf['aum_crore'] < 0,
    'morningstar_rating': (perf['morningstar_rating'] < 1) | (perf['morningstar_rating'] > 5),
    'expense_ratio_pct': (perf['expense_ratio_pct'] < 0) | (perf['expense_ratio_pct'] > 100),
}

flags['hard_invalid'] = False
hard_counts = {}

for col, mask in hard_masks.items():
    hard_counts[col] = int(mask.sum())
    print(col, mask.sum())
    flags['hard_invalid'] = flags['hard_invalid'] | mask
    perf.loc[mask, col] = np.nan

add_log('hard rules applied (rows kept)', perf)

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0
benchmark_3yr_pct 0
max_drawdown_pct 0
std_dev_ann_pct 0
aum_crore 0
morningstar_rating 0
expense_ratio_pct 0


In [8]:
exp = perf['expense_ratio_pct']
print(exp.describe().round(3))

count    40.000
mean      1.237
std       0.387
min       0.550
25%       0.788
50%       1.425
75%       1.540
max       1.640
Name: expense_ratio_pct, dtype: float64


In [9]:
EXP_MIN = 0.1
EXP_MAX = 2.5

below = exp < EXP_MIN
above = exp > EXP_MAX
in_range = exp.between(EXP_MIN, EXP_MAX)
missing = exp.isnull()

print('In range:', in_range.sum())
print('Below', EXP_MIN, ':', below.sum())
print('Above', EXP_MAX, ':', above.sum())
print('Missing:', missing.sum())

assert in_range.sum() + below.sum() + above.sum() + missing.sum() == len(perf), 'Counts do not add up'

In range: 40
Below 0.1 : 0
Above 2.5 : 0
Missing: 0


In [10]:
perf['expense_status'] = np.select(
    [missing, below, above],
    ['missing', 'below range', 'above range'],
    default='in range',
)

flags['expense_out_of_range'] = below | above

print(perf['expense_status'].value_counts())
print()
print(pd.crosstab(perf['plan'], perf['expense_status']))
print()
print(pd.crosstab(perf['category'], perf['expense_status']))

expense_status
in range    40
Name: count, dtype: int64

expense_status  in range
plan                    
Direct                 8
Regular               32

expense_status   in range
category                 
ELSS                    1
Flexi Cap               2
Gilt                    2
Index                   1
Index/ETF               1
Large & Mid Cap         1
Large Cap              14
Liquid                  3
Mid Cap                 7
Short Duration          1
Small Cap               6
Value                   1


In [11]:
n_neg = (perf['max_drawdown_pct'] < 0).sum()
n_pos = (perf['max_drawdown_pct'] > 0).sum()
print('Negative:', n_neg, '| Positive:', n_pos)

if n_neg >= n_pos:
    flags['drawdown_sign_odd'] = perf['max_drawdown_pct'] > 0
else:
    flags['drawdown_sign_odd'] = perf['max_drawdown_pct'] < 0

print('Rows with the odd sign:', flags['drawdown_sign_odd'].sum())

Negative: 40 | Positive: 0
Rows with the odd sign: 0


In [12]:
flag_cols = list(flags.columns)

for col in flag_cols:
    perf['flag_' + col] = flags[col]

perf['anomaly_count'] = flags.sum(axis=1)

print(flags.sum().sort_values(ascending=False))
print()
print('Rows with at least one flag:', (perf['anomaly_count'] > 0).sum(), 'of', len(perf))

not_number              0
hard_invalid            0
expense_out_of_range    0
drawdown_sign_odd       0
dtype: int64

Rows with at least one flag: 0 of 40


In [15]:
review_cols = ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'anomaly_count']
for col in flag_cols:
    review_cols.append('flag_' + col)

review = perf[perf['anomaly_count'] > 0][review_cols].sort_values('anomaly_count', ascending=False)
print(review.shape)
review.head(10)

(0, 10)


,amfi_code,scheme_name,fund_house,category,plan,anomaly_count,flag_not_number,flag_hard_invalid,flag_expense_out_of_range,flag_drawdown_sign_odd


In [13]:
for col in NUM_COLS:
    assert pd.api.types.is_numeric_dtype(perf[col]), col + ' is not numeric'

assert (perf[RETURN_COLS].min() >= -100).all(), 'A return is below -100'
assert perf['expense_ratio_pct'].dropna().between(0, 100).all(), 'Expense ratio outside 0 to 100'
assert len(perf) == n_start, 'The number of rows changed'

print('All checks passed')

All checks passed


In [17]:
perf.to_csv(os.path.join(PROCESSED_DIR, 'scheme_performance_clean.csv'), index=False)
review.to_csv(os.path.join(REPORTS_DIR, 'scheme_performance_review_list.csv'), index=False)

lines = [
    '# Scheme performance cleaning summary',
    '',
    f'- Rows: {n_start} at the start and {len(perf)} at the end. No row was removed.',
    f'- Values that were not numbers, by column: {not_number_counts}.',
    f'- Hard-rule values changed to missing, by column: {hard_counts}.',
    f'- Expense ratio, rule 0.1% to 2.5%: {perf["expense_status"].value_counts().to_dict()}.',
    f'- Flags (number of rows): {flags.sum().to_dict()}.',
    f'- Rows with at least one flag: {(perf["anomaly_count"] > 0).sum()} of {len(perf)}. These are in scheme_performance_review_list.csv.',
    '- Final checks passed: numeric columns, returns not below -100, expense ratio 0 to 100, row count.',
    '',
    '## Assumptions to confirm with my manager',
    '- The range 0.1% to 2.5% applies to all categories and plans. Index funds and liquid funds can be cheaper.',
    '- Return, expense ratio, and drawdown values are in percent, not in fractions.',
    '- A return below -100% cannot be true, so it was changed to missing.',
    '- Flagged rows were kept. A person decides what to do with them.',
]

text = '\n'.join(lines)

with open(os.path.join(REPORTS_DIR, 'scheme_performance_cleaning_summary.md'), 'w', encoding='utf-8') as f:
    f.write(text)

print(text)

# Scheme performance cleaning summary

- Rows: 40 at the start and 40 at the end. No row was removed.
- Values that were not numbers, by column: {'return_1yr_pct': 0, 'return_3yr_pct': 0, 'return_5yr_pct': 0, 'benchmark_3yr_pct': 0, 'alpha': 0, 'beta': 0, 'sharpe_ratio': 0, 'sortino_ratio': 0, 'std_dev_ann_pct': 0, 'max_drawdown_pct': 0, 'aum_crore': 0, 'expense_ratio_pct': 0, 'morningstar_rating': 0}.
- Hard-rule values changed to missing, by column: {'return_1yr_pct': 0, 'return_3yr_pct': 0, 'return_5yr_pct': 0, 'benchmark_3yr_pct': 0, 'max_drawdown_pct': 0, 'std_dev_ann_pct': 0, 'aum_crore': 0, 'morningstar_rating': 0, 'expense_ratio_pct': 0}.
- Expense ratio, rule 0.1% to 2.5%: {'in range': 40}.
- Flags (number of rows): {'not_number': 0, 'hard_invalid': 0, 'expense_out_of_range': 0, 'drawdown_sign_odd': 0}.
- Rows with at least one flag: 0 of 40. These are in scheme_performance_review_list.csv.
- Final checks passed: numeric columns, returns not below -100, expense ratio 0 to 100,